In [1]:
import triton
import triton.language as tl
import torch

In [15]:
X = torch.randint(-128, 127, (128, 32), dtype=torch.int8).cuda()
W = torch.randint(-128, 127, (64,32), dtype=torch.int8).cuda()

In [29]:
@triton.jit
def int8_gemm_kernel(
    X_ptr,
    W_ptr,
    Y_ptr,
    M,
    N,
    K,
    stride_xm,
    stride_xk,
    stride_wn,
    stride_wk,
    stride_ym,
    stride_yn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    ):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.int32)

    for k in range(0, K, BLOCK_K):
        k_offsets = k + offs_k
        x_ptrs = X_ptr + (offs_m[:, None] * stride_xm + k_offsets[None, :] * stride_xk)
        w_ptrs = W_ptr + (offs_n[None, :] * stride_wn + k_offsets[:, None] * stride_wk)

        x_block = tl.load(
            x_ptrs,
            mask=(offs_m[:, None] < M) & (k_offsets[None, :] < K),
            other=0,
        )
        w_block = tl.load(
            w_ptrs,
            mask=(offs_n[None, :] < N) & (k_offsets[:, None] < K),
            other=0,
        )

        acc += tl.dot(x_block, w_block)

    y_ptrs = Y_ptr + (offs_m[:, None] * stride_ym + offs_n[None, :] * stride_yn)
    tl.store(
        y_ptrs,
        acc,
        mask=(offs_m[:, None] < M) & (offs_n[None, :] < N),
    )

In [40]:
expected = X.int().cpu().mm(W.t().cpu().int()).cuda()
expected

tensor([[  5630,  24089, -56637,  ...,  -4544,  11608,   2516],
        [ 14968,  -6997,   2034,  ...,  32094, -16340,   7217],
        [  1987,   6822,  70257,  ..., -18964,  39127,  17374],
        ...,
        [  6591,   7244,  13074,  ..., -43289,  25144, -21897],
        [-23020, -33460, -34200,  ...,  23408,  77944,  33442],
        [ 38718, -29144,  21787,  ..., -18493, -40812, -33345]],
       device='cuda:0', dtype=torch.int32)

In [36]:
y = torch.zeros((128, 64), dtype=torch.int32).cuda()
B_M, B_N, B_K = 32, 32, 32
grid = (triton.cdiv(X.shape[0], B_M), triton.cdiv(W.shape[0], B_N))
int8_gemm_kernel[grid](
    X,
    W,
    y,
    X.shape[0],
    W.shape[0],
    X.shape[1],
    X.stride(0),
    X.stride(1),
    W.stride(0),
    W.stride(1),
    y.stride(0),
    y.stride(1),
    B_M,
    B_N,
    B_K,
 )

In [10]:
import torch
import triton
import triton.language as tl

# -----------------------------------------------------------------------------
# 1. ASM String Generator (Run once in Python)
# -----------------------------------------------------------------------------
def generate_fp4_decode_asm():
    """
    Generates the PTX assembly string to decode 4 packed bytes (8 FP4 codes)
    into 8 int8 values packed into two 32-bit registers.
    """
    asm = """
    .reg .b32 r_in, r_lut;
    .reg .b32 r_idx, r_sign, r_shift, r_mag, r_neg, r_val, r_tmp;
    .reg .b32 r_out0, r_out1;
    .reg .pred p_neg;

    mov.b32 r_in, $2;             // Load Input (32 bits = 4 packed bytes)
    mov.b32 r_lut, 0xC8643210;    // LUT: 12, 8, 6, 4, 3, 2, 1, 0
    mov.b32 r_out0, 0;            // Init Output 0
    mov.b32 r_out1, 0;            // Init Output 1
    """

    # --- Group 1: Output Register 0 (Decodes Input Bytes 0 & 1) ---
    # Decodes nibbles 0, 1, 2, 3 (Bits 0-15 of input)
    for i in range(4):
        start_bit = i * 4
        shift_pack = i * 8
        asm += f"""
    bfe.u32 r_idx,  r_in, {start_bit}, 3;     // Extract 3-bit Mag Index
    bfe.u32 r_sign, r_in, {start_bit+3}, 1;   // Extract Sign Bit
    shl.b32 r_shift, r_idx, 2;                // Shift for LUT
    bfe.u32 r_mag, r_lut, r_shift, 4;         // LUT Lookup
    neg.s32 r_neg, r_mag;                     // Negate
    setp.ne.u32 p_neg, r_sign, 0;             // Check Sign
    selp.s32 r_val, r_neg, r_mag, p_neg;      // Select Value
    and.b32 r_val, r_val, 0xFF;               // Mask to 8 bits
    shl.b32 r_tmp, r_val, {shift_pack};       // Shift to packed position
    or.b32  r_out0, r_out0, r_tmp;            // Accumulate
        """

    # --- Group 2: Output Register 1 (Decodes Input Bytes 2 & 3) ---
    # Decodes nibbles 4, 5, 6, 7 (Bits 16-31 of input)
    for i in range(4, 8):
        start_bit = i * 4
        shift_pack = (i - 4) * 8
        asm += f"""
    bfe.u32 r_idx,  r_in, {start_bit}, 3;
    bfe.u32 r_sign, r_in, {start_bit+3}, 1;
    shl.b32 r_shift, r_idx, 2;
    bfe.u32 r_mag, r_lut, r_shift, 4;
    neg.s32 r_neg, r_mag;
    setp.ne.u32 p_neg, r_sign, 0;
    selp.s32 r_val, r_neg, r_mag, p_neg;
    and.b32 r_val, r_val, 0xFF;
    shl.b32 r_tmp, r_val, {shift_pack};
    or.b32  r_out1, r_out1, r_tmp;
        """

    # Write to output registers ($0 and $1)
    asm += """
    mov.b32 $0, r_out0;
    mov.b32 $1, r_out1;
    """
    return asm

# Pre-generate the string to ensure it's a compile-time constant for the JIT
FP4_DECODE_PTX = generate_fp4_decode_asm()



In [14]:
print(rf"{FP4_DECODE_PTX}")


    .reg .b32 r_in, r_lut;
    .reg .b32 r_idx, r_sign, r_shift, r_mag, r_neg, r_val, r_tmp;
    .reg .b32 r_out0, r_out1;
    .reg .pred p_neg;

    mov.b32 r_in, $2;             // Load Input (32 bits = 4 packed bytes)
    mov.b32 r_lut, 0xC8643210;    // LUT: 12, 8, 6, 4, 3, 2, 1, 0
    mov.b32 r_out0, 0;            // Init Output 0
    mov.b32 r_out1, 0;            // Init Output 1
    
    bfe.u32 r_idx,  r_in, 0, 3;     // Extract 3-bit Mag Index
    bfe.u32 r_sign, r_in, 3, 1;   // Extract Sign Bit
    shl.b32 r_shift, r_idx, 2;                // Shift for LUT
    bfe.u32 r_mag, r_lut, r_shift, 4;         // LUT Lookup
    neg.s32 r_neg, r_mag;                     // Negate
    setp.ne.u32 p_neg, r_sign, 0;             // Check Sign
    selp.s32 r_val, r_neg, r_mag, p_neg;      // Select Value
    and.b32 r_val, r_val, 0xFF;               // Mask to 8 bits
    shl.b32 r_tmp, r_val, 0;       // Shift to packed position
    or.b32  r_out0, r_out0, r_tmp;            // Accumulate


In [ ]:
# -----------------------------------------------------------------------------
# 2. Triton Kernel
# -----------------------------------------------------------------------------
@triton.jit
def fp4_decode_kernel(
    input_ptr,
    output_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    pid = tl.program_id(axis=0)

    # Offsets for uint8 data
    ofs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = ofs < n_elements

    # Load uint8 data
    # pack=4 in inline_asm will treat this as vectors of 4 bytes
    packed_input = tl.load(input_ptr + ofs, mask=mask)

    # Call Inline ASM
    # Input: 1 vector (4 bytes)
    # Output: 2 vectors (4 int8s each)
    # constraints: "=r,=r,r" -> 2 outputs, 1 input
    # pack=4: The instruction operates on 4 elements at a time.
    out0 = tl.inline_asm_elementwise(
        asm="""
    .reg .b32 r_in, r_lut;
    .reg .b32 r_idx, r_sign, r_shift, r_mag, r_neg, r_val, r_tmp;
    .reg .b32 r_out0, r_out1;
    .reg .pred p_neg;

    mov.b32 r_in, $2;             // Load Input (32 bits = 4 packed bytes)
    mov.b32 r_lut, 0xC8643210;    // LUT: 12, 8, 6, 4, 3, 2, 1, 0
    mov.b32 r_out0, 0;            // Init Output 0
    mov.b32 r_out1, 0;            // Init Output 1
    
    bfe.u32 r_idx,  r_in, 0, 3;     // Extract 3-bit Mag Index
    bfe.u32 r_sign, r_in, 3, 1;   // Extract Sign Bit
    shl.b32 r_shift, r_idx, 2;                // Shift for LUT
    bfe.u32 r_mag, r_lut, r_shift, 4;         // LUT Lookup
    neg.s32 r_neg, r_mag;                     // Negate
    setp.ne.u32 p_neg, r_sign, 0;             // Check Sign
    selp.s32 r_val, r_neg, r_mag, p_neg;      // Select Value
    and.b32 r_val, r_val, 0xFF;               // Mask to 8 bits
    shl.b32 r_tmp, r_val, 0;       // Shift to packed position
    or.b32  r_out0, r_out0, r_tmp;            // Accumulate
        
    bfe.u32 r_idx,  r_in, 4, 3;     // Extract 3-bit Mag Index
    bfe.u32 r_sign, r_in, 7, 1;   // Extract Sign Bit
    shl.b32 r_shift, r_idx, 2;                // Shift for LUT
    bfe.u32 r_mag, r_lut, r_shift, 4;         // LUT Lookup
    neg.s32 r_neg, r_mag;                     // Negate
    setp.ne.u32 p_neg, r_sign, 0;             // Check Sign
    selp.s32 r_val, r_neg, r_mag, p_neg;      // Select Value
    and.b32 r_val, r_val, 0xFF;               // Mask to 8 bits
    shl.b32 r_tmp, r_val, 8;       // Shift to packed position
    or.b32  r_out0, r_out0, r_tmp;            // Accumulate
        
    bfe.u32 r_idx,  r_in, 8, 3;     // Extract 3-bit Mag Index
    bfe.u32 r_sign, r_in, 11, 1;   // Extract Sign Bit
    shl.b32 r_shift, r_idx, 2;                // Shift for LUT
    bfe.u32 r_mag, r_lut, r_shift, 4;         // LUT Lookup
    neg.s32 r_neg, r_mag;                     // Negate
    setp.ne.u32 p_neg, r_sign, 0;             // Check Sign
    selp.s32 r_val, r_neg, r_mag, p_neg;      // Select Value
    and.b32 r_val, r_val, 0xFF;               // Mask to 8 bits
    shl.b32 r_tmp, r_val, 16;       // Shift to packed position
    or.b32  r_out0, r_out0, r_tmp;            // Accumulate
        
    bfe.u32 r_idx,  r_in, 12, 3;     // Extract 3-bit Mag Index
    bfe.u32 r_sign, r_in, 15, 1;   // Extract Sign Bit
    shl.b32 r_shift, r_idx, 2;                // Shift for LUT
    bfe.u32 r_mag, r_lut, r_shift, 4;         // LUT Lookup
    neg.s32 r_neg, r_mag;                     // Negate
    setp.ne.u32 p_neg, r_sign, 0;             // Check Sign
    selp.s32 r_val, r_neg, r_mag, p_neg;      // Select Value
    and.b32 r_val, r_val, 0xFF;               // Mask to 8 bits
    shl.b32 r_tmp, r_val, 24;       // Shift to packed position
    or.b32  r_out0, r_out0, r_tmp;            // Accumulate
        
    bfe.u32 r_idx,  r_in, 16, 3;
    bfe.u32 r_sign, r_in, 19, 1;
    shl.b32 r_shift, r_idx, 2;
    bfe.u32 r_mag, r_lut, r_shift, 4;
    neg.s32 r_neg, r_mag;
    setp.ne.u32 p_neg, r_sign, 0;
    selp.s32 r_val, r_neg, r_mag, p_neg;
    and.b32 r_val, r_val, 0xFF;
    shl.b32 r_tmp, r_val, 0;
    or.b32  r_out1, r_out1, r_tmp;
        
    bfe.u32 r_idx,  r_in, 20, 3;
    bfe.u32 r_sign, r_in, 23, 1;
    shl.b32 r_shift, r_idx, 2;
    bfe.u32 r_mag, r_lut, r_shift, 4;
    neg.s32 r_neg, r_mag;
    setp.ne.u32 p_neg, r_sign, 0;
    selp.s32 r_val, r_neg, r_mag, p_neg;
    and.b32 r_val, r_val, 0xFF;
    shl.b32 r_tmp, r_val, 8;
    or.b32  r_out1, r_out1, r_tmp;
        
    bfe.u32 r_idx,  r_in, 24, 3;
    bfe.u32 r_sign, r_in, 27, 1;
    shl.b32 r_shift, r_idx, 2;
    bfe.u32 r_mag, r_lut, r_shift, 4;
    neg.s32 r_neg, r_mag;
    setp.ne.u32 p_neg, r_sign, 0;
    selp.s32 r_val, r_neg, r_mag, p_neg;
    and.b32 r_val, r_val, 0xFF;
    shl.b32 r_tmp, r_val, 16;
    or.b32  r_out1, r_out1, r_tmp;
        
    bfe.u32 r_idx,  r_in, 28, 3;
    bfe.u32 r_sign, r_in, 31, 1;
    shl.b32 r_shift, r_idx, 2;
    bfe.u32 r_mag, r_lut, r_shift, 4;
    neg.s32 r_neg, r_mag;
    setp.ne.u32 p_neg, r_sign, 0;
    selp.s32 r_val, r_neg, r_mag, p_neg;
    and.b32 r_val, r_val, 0xFF;
    shl.b32 r_tmp, r_val, 24;
    or.b32  r_out1, r_out1, r_tmp;
        
    mov.b32 $0, r_out0;
    mov.b32 $1, r_out1;
""",
        constraints="=r,=r,r",
        args=[packed_input],
        dtype=tl.int8,
        is_pure=True,
        pack=4,
    )

    # At this point:
    # out0 is a Tensor[int8] of shape (BLOCK_SIZE) containing the first half of decoded values
    # out1 is a Tensor[int8] of shape (BLOCK_SIZE) containing the second half
    # They are ready for int8 matmul or storage.

    # Example Storage Logic (Interleaved):
    # We store out0 at 2*i and out1 at 2*i+1.
    # To do this efficiently, we can't easily interleave scalar by scalar with just stores.
    # But if we treat output as int32 pointer, we can store the packed registers directly.

    # However, to demonstrate they are standard int8 tensors, we can store them like this:
    # (Note: simpler for verification, slightly less efficient than vector store)

    # Create interleaved offsets
    ofs_out_0 = 2 * pid * BLOCK_SIZE + tl.arange(0, 2 * BLOCK_SIZE)
    tl.store(output_ptr + ofs_out_0, out0, mask=(ofs_out_0 < 2 * n_elements))

In [26]:
test_fp4()

CompilationError: at 147:4:

    # Example Storage Logic (Interleaved):
    # We store out0 at 2*i and out1 at 2*i+1.
    # To do this efficiently, we can't easily interleave scalar by scalar with just stores.
    # But if we treat output as int32 pointer, we can store the packed registers directly.

    # However, to demonstrate they are standard int8 tensors, we can store them like this:
    # (Note: simpler for verification, slightly less efficient than vector store)

    # Create interleaved offsets
    ofs_out_0 = 2 * pid * BLOCK_SIZE + tl.arange(0, 2 * BLOCK_SIZE)
    tl.store(output_ptr + ofs_out_0, out0, mask=(ofs_out_0 < 2 * n_elements))
    ^
Cannot broadcast, the expanded size of the tensor (512) must match the existing size (256) at non-singleton dimension 0: ['256'], ['512']